# USA — Already-on-BFP APT Validation

## Purpose

Validate a set of **scoped APT ids** (USA) against the current layer `14533` and today's
Building Footprint (BFP), then emit the Data Correctness (DC) input format for the ones that
qualify.

## Steps

1. **Load snapshot for layer `14533`** and filter to USA address points; display the row count. Dataset name: `latestAptDS`.
2. **Load BFP** from catalog `preprocess_prod.bfp`. Dataset name: `usaBfpDataset`.
3. **Load the scoped CSV file(s)** as `scopedDS`.
4. **Join `scopedDS` with `latestAptDS`** on the APT ids → `joinedDS`.
5. **Enrich `joinedDS`:** add `latest_vs_scoped_distance_in_meters` (distance between the `latestAptDS` point and the `scopedDS` point) and `location_change` (true/false, whether the geometry differs between latest and scoped).
6. **Join `joinedDS` with `usaBfpDataset`:** add `is_intersects_with_bfp` (true if the APT intersects any BFP) and `bfp_geom` (the intersecting BFP polygon).
7. **Filter** to rows that intersect a BFP.
8. **Format** the scoped output in the Data Correctness (DC) job input format.

In [ ]:
%sql
DROP DATABASE IF EXISTS usa_alredy_on_bfp_validation CASCADE;
CREATE DATABASE IF NOT EXISTS usa_alredy_on_bfp_validation;

In [ ]:
%scala
// Shared constants used across the cells below (defined once here; values do not change between runs).
val LAYER_ID = 14533
val COUNTRY_ISO = "USA"
val NEW_DATABASE = "usa_alredy_on_bfp_validation"
val latestSnapshotTable = s"${NEW_DATABASE}.apt_snapshot_new"

In [ ]:
%scala
// The newest revision is resolved automatically via LoadSnapshotInfo.getLatestRevisionId.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(LAYER_ID)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(NEW_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded latest snapshot, keeping only APTs whose metadata:country tag = USA
val latestAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
latestAptDS.write.format("delta").mode("overwrite").saveAsTable(latestSnapshotTable)

In [ ]:
%scala
println(s"LATEST snapshot written to: $latestSnapshotTable")

In [ ]:
%scala
// Read the persisted USA APT snapshot back from the catalog as usaAptDataset.
// Table: usa_alredy_on_bfp_validation.apt_snapshot_new (written in the previous cell).

import org.apache.spark.sql.{Dataset, Encoders}
import com.tomtom.orbis.io.spark.model.OrbisElement

val usaAptDataset: Dataset[OrbisElement] =
  spark.table(latestSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

display(usaAptDataset)

In [ ]:
%scala
println(s"usaAptDataset read from: $latestSnapshotTable, count: ${usaAptDataset.count()}")

In [ ]:
%scala
// Read the USA Building Footprint (BFP) polygons from preprocess_prod.bfp.

import org.apache.spark.sql.functions._
import org.apache.sedona.spark.SedonaContext

// Sedona is required for the ST_Intersects spatial join in the next cell
val sedona = SedonaContext.create(spark)

val usaBfpDataset = spark.table("preprocess_prod.bfp")
  .filter(col("licenseZone") === COUNTRY_ISO)
  .filter(col("wkt").isNotNull)
  .withColumn("bfp_geom", expr("ST_GeomFromWKT(wkt)"))

display(usaBfpDataset)

In [ ]:
%scala
println(s"USA BFP polygon count: ${usaBfpDataset.count()}")

In [ ]:
%scala
// Step 3 - Load the scoped APT CSV as scopedDS.
// CSV columns: Product_orbis_id (14533_high_low, unsigned), Target_Value_old_xy ("POINT (lon lat)"),
// Relocation_Type, old_address_point_type, new_address_point_type, cntry_code, addr_state.

import org.apache.spark.sql.functions._

// >>> Set the scoped-input CSV path here <<<
val scopedCsvPath = "/mnt/aqua/data-correctness/correction-input/SEACO-6267-USA-Already-On-BFP-Scope/ALREADY_ON_CORRECT_BFP/WI_ALREADY_ON_CORRECT_BFP.csv"

val scopedDS = spark.read.option("header", "true").csv(scopedCsvPath)
  // Target_Value_old_xy = "POINT (lon lat)" -> scoped_lon is the first coord (X), scoped_lat the second (Y)
  .withColumn("scoped_lon",
    regexp_extract(col("Target_Value_old_xy"), """POINT \(([^ ]+) ([^)]+)\)""", 1).cast("double"))
  .withColumn("scoped_lat",
    regexp_extract(col("Target_Value_old_xy"), """POINT \(([^ ]+) ([^)]+)\)""", 2).cast("double"))
  // canonical POINT(lon lat) (no space) so it lines up with the latest_wkt built in Step 4
  .withColumn("scoped_wkt", concat(lit("POINT("), col("scoped_lon"), lit(" "), col("scoped_lat"), lit(")")))

display(scopedDS)

In [ ]:
%scala
println(s"scopedDS count: ${scopedDS.count()}")

In [ ]:
%scala
// Step 4 - Join scopedDS with the loaded snapshot (usaAptDataset) on the APT id -> joinedDS.
// The snapshot ids are colon-form (14533:high:low); derive an underscore Product_orbis_id
// (14533_high_low) on the snapshot side so it matches scopedDS.Product_orbis_id directly.

import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.Id
import java.lang.{Long => JLong}

// Build the underscore-separated unsigned id: {layerId}_{unsignedHigh}_{unsignedLow}
def convertOrbisIdToProductId(orbisId: Id): String = {
  Seq(orbisId.layerId.getOrElse(LAYER_ID).toString,
      JLong.toUnsignedString(orbisId.high),
      JLong.toUnsignedString(orbisId.low)).mkString("_")
}
val convertOrbisIdToProductIdUDF = udf((orbisId: Id) => convertOrbisIdToProductId(orbisId))

// Reduce the loaded snapshot (usaAptDataset) to the join key + location + tags.
val latestForJoin = usaAptDataset
  .withColumn("Product_orbis_id", convertOrbisIdToProductIdUDF(col("id")))
  .withColumnRenamed("lat", "latest_lat")
  .withColumnRenamed("lng", "latest_lon")
  .withColumnRenamed("tags", "latest_tags")
  .withColumn("latest_wkt", concat(lit("POINT("), col("latest_lon"), lit(" "), col("latest_lat"), lit(")")))
  .select("Product_orbis_id", "latest_lat", "latest_lon", "latest_wkt", "latest_tags")

// Keep all scoped rows; bring in the snapshot location (latest_* null where id absent in snapshot).
val joinedDS = scopedDS.join(latestForJoin, Seq("Product_orbis_id"), "left")

// Scoped APTs that no longer exist in the latest snapshot (id present in scopedDS, absent in latestForJoin).
val scopedNotExistDS = scopedDS.join(latestForJoin, Seq("Product_orbis_id"), "left_anti")

display(joinedDS)

In [ ]:
%scala
println(s"scopedDS: ${scopedDS.count()}, matched in snapshot: ${joinedDS.filter(col("latest_wkt").isNotNull).count()}, " +
  s"not in snapshot (scopedNotExistDS): ${scopedNotExistDS.count()}")

In [ ]:
%scala
// Display the scoped APTs that no longer exist in the latest snapshot.
display(scopedNotExistDS)

In [ ]:
%scala
// Step 5 - Enrich joinedDS with latest_vs_scoped_distance_in_meters and location_change.
// distance_in_meters: great-circle (Haversine) distance between the latest point and the scoped point.
// location_change: true when the latest geometry differs from the scoped geometry.

import org.apache.spark.sql.functions._

// Haversine distance in meters (mean earth radius) - avoids Sedona for the distance calc
val EARTH_RADIUS_M = 6371008.8
val dLat = radians(col("latest_lat") - col("scoped_lat"))
val dLon = radians(col("latest_lon") - col("scoped_lon"))
val haversine =
  pow(sin(dLat / 2), 2) +
  cos(radians(col("scoped_lat"))) * cos(radians(col("latest_lat"))) * pow(sin(dLon / 2), 2)
val distanceMeters = lit(2 * EARTH_RADIUS_M) * asin(sqrt(haversine))

val joinedWithDistance = joinedDS
  .withColumn("latest_vs_scoped_distance_in_meters",
    when(col("latest_lat").isNotNull && col("scoped_lat").isNotNull, distanceMeters))
  // geometry changed when the latest point's coordinates differ from the scoped point's
  .withColumn("location_change",
    when(col("latest_lat").isNull || col("scoped_lat").isNull, lit(null).cast("boolean"))
      .otherwise(!(col("latest_lat") === col("scoped_lat") && col("latest_lon") === col("scoped_lon"))))

display(joinedWithDistance)

In [ ]:
%scala
println(s"joinedWithDistance rows: ${joinedWithDistance.count()}, " +
  s"location_change=true: ${joinedWithDistance.filter(col("location_change") === true).count()}")

In [ ]:
%scala
// Step 6 - Join joinedWithDistance with usaBfpDataset; add is_intersects_with_bfp and bfp_geom.
// The latest APT point is tested against the BFP polygons (ST_Intersects). An INNER spatial join
// (Sedona partitioned RangeJoin, no broadcast) finds the matched footprint, then a cheap equi LEFT
// join back keeps APTs that fall outside every footprint with is_intersects_with_bfp = false.

import org.apache.spark.sql.functions._

// A spatial LEFT join is NOT optimized by Sedona -> Spark broadcasts the whole BFP relation
// (EXECUTOR_BROADCAST_JOIN_OOM). Disable the threshold so the inner join is a Sedona RangeJoin.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

// BFP polygons prepared once for the spatial join (bfp_geom carries the matched footprint WKT)
val bfpForJoin = usaBfpDataset
  .select(col("bfpId").as("bfp_id"), col("wkt").as("bfp_geom_wkt"), col("bfp_geom"))

// Build the latest APT point (skip rows with no latest location -> nothing to intersect)
val aptPoints = joinedWithDistance
  .filter(col("latest_lat").isNotNull && col("latest_lon").isNotNull)
  .withColumn("apt_geom", expr("ST_GeomFromText(concat('POINT(', latest_lon, ' ', latest_lat, ')'))"))

// INNER spatial join -> Sedona RangeJoin (no broadcast). Keep one footprint per APT.
val matched = aptPoints.alias("apt")
  .join(bfpForJoin.alias("bfp"), expr("ST_Intersects(bfp.bfp_geom, apt.apt_geom)"))
  .select(
    col("apt.Product_orbis_id").as("Product_orbis_id"),
    col("bfp.bfp_id").as("bfp_id"),
    col("bfp.bfp_geom_wkt").as("bfp_geom"))
  .dropDuplicates("Product_orbis_id")

// Equi LEFT join back onto the full set (no spatial predicate -> cheap shuffle join)
val joinedWithBfp = joinedWithDistance
  .join(matched, Seq("Product_orbis_id"), "left")
  .withColumn("is_intersects_with_bfp", col("bfp_id").isNotNull)

display(joinedWithBfp)

In [ ]:
%scala
// Step 7 - Split the BFP-checked set: intersectingScope (latest location on a BFP, the validated
// "already-on-BFP" set) and notIntersectingScope (latest location not on any BFP).
import org.apache.spark.sql.functions._

// intersectingScope is reused by the DC-format write below, so cache it.
val intersectingScope = joinedWithBfp.filter(col("is_intersects_with_bfp") === true).cache()
val notIntersectingScope = joinedWithBfp.filter(col("is_intersects_with_bfp") === false)

display(intersectingScope)

In [ ]:
%scala
// Display the APTs whose latest location does NOT intersect any BFP.
display(notIntersectingScope)

In [ ]:
%scala
println(s"joinedWithBfp: ${joinedWithBfp.count()}, " +
  s"intersecting a BFP (intersectingScope): ${intersectingScope.count()}, " +
  s"not intersecting (notIntersectingScope): ${notIntersectingScope.count()}")

In [ ]:
%scala
// Step 8 - Format the BFP-intersecting scope in the Data Correctness (DC) job input format and
// write it as a single CSV. Mirrors the DC-input format produced by the RoofTop scoping notebook.
// Also write the not-on-BFP scope (notIntersectingScope) to its own CSV.

import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions._

val dcInputScope = intersectingScope
  // Product_orbis_id is already in 14533_high_low form (carried from the scoped CSV)
  .withColumn("Product_Address_component", lit("metadata:apa:previous_location"))
  // target_value: the scoped target point from the input CSV
  .withColumn("target_value", col("Target_Value_old_xy"))
  // Orbis_X / Orbis_Y parsed from latest_wkt = POINT(X Y): X is the first coord (lon), Y the second (lat)
  .withColumn("Orbis_Y", regexp_extract(col("latest_wkt"), """POINT\(([^ ]+) ([^)]+)\)""", 1))
  .withColumn("Orbis_X", regexp_extract(col("latest_wkt"), """POINT\(([^ ]+) ([^)]+)\)""", 2))
  // Relocation_Type carried over from the scoped CSV (e.g. ALREADY_ON_CORRECT_BFP)
  .withColumn("Relocation_Type", col("Relocation_Type"))
  .select("Product_orbis_id", "Product_Address_component", "target_value",
          "Orbis_X", "Orbis_Y", "Relocation_Type")

// not-on-BFP scope: CSV cannot serialize array<struct>, so emit latest_tags as a JSON string.
val notOnBfpScope = notIntersectingScope
  .withColumn("latest_tags", to_json(col("latest_tags")))

// >>> Set the output directory here <<<
val outputDir = "/mnt/aqua/data-correctness/correction-input/SEACO-6267-USA-Already-On-BFP-Scope"

// Output file names derived from the scoped input file name (scopedCsvPath).
// e.g. WI_ALREADY_ON_CORRECT_BFP.csv -> WI_ALREADY_ON_CORRECT_BFP-Already-On-BFP-DC-Input-format.csv
val scopedFileBaseName = scopedCsvPath.split("/").last.stripSuffix(".csv")
val dcInputFileName = s"$scopedFileBaseName-Already-On-BFP-DC-Input-format.csv"
val notOnBfpFileName = s"$scopedFileBaseName-not_of_bfp_scope.csv"

// Write a DataFrame as one named .csv file: Spark emits a folder of part files, so coalesce to a
// single partition, then promote that part file to the target name and clean up the temp dir.
def writeSingleCsv(df: DataFrame, fileName: String): Unit = {
  val target = s"$outputDir/$fileName"
  val tmpDir = s"$outputDir/${fileName}_tmp"
  df.coalesce(1).write.option("header", "true").mode("overwrite").csv(tmpDir)
  val partFile = dbutils.fs.ls(tmpDir).filter(_.name.endsWith(".csv")).head.path
  dbutils.fs.rm(target)
  dbutils.fs.cp(partFile, target)
  dbutils.fs.rm(tmpDir, recurse = true)
  println(s"CSV written to: $target")
}

writeSingleCsv(dcInputScope, dcInputFileName)
writeSingleCsv(notOnBfpScope, notOnBfpFileName)
display(dcInputScope)

In [ ]:
%scala
println(s"DC input scope count: ${dcInputScope.count()}")